# Create a table with an identity column

https://learn.microsoft.com/en-us/fabric/data-engineering/delta-lake-generated-columns?tabs=pyspark&source=docs#create-a-table-with-an-identity-column

- Let's build a real-world Sales model in a Fabric Lakehouse with:
    - DimCustomer (Identity SK)
    - DimProduct (Identity SK)
    - FactSales (stores SKs)

- SCD Type 1 MERGE for dimensions
- Incremental sales load
- This is a pattern you'll actually use in a data warehouse.

```
Source Files
    |
    v

stg_customer
stg_product
stg_sales

    |
    v

DimCustomer
(CustomerSK Identity)

DimProduct
(ProductSK Identity)

    |
    v

FactSales
(CustomerSK, ProductSK)
```

## Step 1: Create Customer Dimension

In [1]:
%%sql

DROP TABLE IF EXISTS dbo.dim_customer;

CREATE TABLE dbo.dim_customer
(
    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID STRING,
    CustomerName STRING,
    City STRING
)
USING DELTA;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 3, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## Step 2: Create Product Dimension

In [2]:
%%sql

DROP TABLE IF EXISTS dbo.dim_product;

CREATE TABLE dbo.dim_product
(
    ProductSK BIGINT GENERATED ALWAYS AS IDENTITY,
    ProductID STRING,
    ProductName STRING,
    Category STRING
)
USING DELTA;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 5, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## Step 3: Create Fact Table
Note: No identity needed. Fact table stores dimension surrogate keys.

In [4]:
%%sql

DROP TABLE IF EXISTS dbo.fact_sales;

CREATE TABLE dbo.fact_sales
(
    SalesDate DATE,
    CustomerSK BIGINT,
    ProductSK BIGINT,
    Quantity INT,
    Amount DECIMAL(18,2)
)
USING DELTA;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 9, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## Step 4: Create Staging Tables

In [6]:
%%sql

CREATE OR REPLACE TABLE dbo.stg_customer
(
    CustomerID STRING,
    CustomerName STRING,
    City STRING
)
USING DELTA;



CREATE OR REPLACE TABLE dbo.stg_product
(
    ProductID STRING,
    ProductName STRING,
    Category STRING
)
USING DELTA;



CREATE OR REPLACE TABLE dbo.stg_sales
(
    SalesDate DATE,
    CustomerID STRING,
    ProductID STRING,
    Quantity INT,
    Amount DECIMAL(18,2)
)
USING DELTA;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 13, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## Step 5: Day-1 Load

In [7]:
%%sql

INSERT INTO dbo.stg_customer VALUES
('C001','ABC Corp','New York'),
('C002','XYZ Corp','Dallas'),
('C003','Global Inc','Atlanta');



INSERT INTO dbo.stg_product VALUES
('P100','Laptop','Electronics'),
('P200','Mouse','Electronics'),
('P300','Chair','Furniture');

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 15, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## Step 6: SCD1 Merge Customer

In [8]:
%%sql

MERGE INTO dbo.dim_customer t
USING dbo.stg_customer s
ON t.CustomerID = s.CustomerID

WHEN MATCHED THEN
UPDATE SET
    t.CustomerName = s.CustomerName,
    t.City = s.City

WHEN NOT MATCHED THEN
INSERT
(
    CustomerID,
    CustomerName,
    City
)
VALUES
(
    s.CustomerID,
    s.CustomerName,
    s.City
);

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 16, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

## Step 7: SCD1 Merge Product

In [9]:
%%sql

MERGE INTO dbo.dim_product t
USING dbo.stg_product s
ON t.ProductID = s.ProductID

WHEN MATCHED THEN
UPDATE SET
    t.ProductName = s.ProductName,
    t.Category = s.Category

WHEN NOT MATCHED THEN
INSERT
(
    ProductID,
    ProductName,
    Category
)
VALUES
(
    s.ProductID,
    s.ProductName,
    s.Category
);

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 17, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

### Check Identity Values

In [10]:
%%sql

SELECT *
FROM dbo.dim_customer
ORDER BY CustomerSK;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 18, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 4 fields>

In [11]:
%%sql

SELECT *
FROM dbo.dim_product
ORDER BY ProductSK;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 19, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 4 fields>

## Step 8: Load Sales

In [12]:
%%sql

INSERT INTO dbo.stg_sales VALUES
('2026-10-20','C001','P100',2,2000),
('2026-10-20','C001','P200',1,20),
('2026-10-20','C002','P300',5,500);

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Step 9: Populate Fact Table
- Lookup surrogate keys.

In [13]:
%%sql

INSERT INTO dbo.fact_sales

SELECT
    s.SalesDate,
    c.CustomerSK,
    p.ProductSK,
    s.Quantity,
    s.Amount
FROM dbo.stg_sales s
JOIN dbo.dim_customer c
    ON s.CustomerID = c.CustomerID
JOIN dbo.dim_product p
    ON s.ProductID = p.ProductID;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### Verify Fact

In [14]:
%%sql

SELECT *
FROM dbo.fact_sales;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 22, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 5 fields>

## Day-2 Incremental Load

In [15]:
%%sql
TRUNCATE TABLE dbo.stg_customer;

INSERT INTO dbo.stg_customer VALUES
('C002','XYZ Corp','Houston'),
('C004','Future Corp','Chicago');

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 24, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [16]:
%%sql

MERGE INTO dbo.dim_customer t
USING dbo.stg_customer s
ON t.CustomerID = s.CustomerID

WHEN MATCHED THEN
UPDATE SET
    t.CustomerName = s.CustomerName,
    t.City = s.City

WHEN NOT MATCHED THEN
INSERT
(
    CustomerID,
    CustomerName,
    City
)
VALUES
(
    s.CustomerID,
    s.CustomerName,
    s.City
);

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [17]:
%%sql
SELECT *
FROM dbo.dim_customer
ORDER BY CustomerSK;

StatementMeta(, 73337ae2-16a2-42dc-8241-9261c6ab3bd3, 26, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 4 fields>